In [ ]:
import torch
from src.plot import Plotter
from src.model_utils import build_model, build_diffuser, load_config
import pathlib

## 条件付き拡散モデルによる生成

In [ ]:
trained_model_folder_name = "CIFAR10_SimpleU-net_20260503"
diffusion_guidance_scale = 8.0

In [ ]:
model_folder_path = pathlib.Path("./models") / trained_model_folder_name / "weights"
model_type = "final" # "final" or "ema" or "checkpoint"
if model_type == "final":
    model_pth_path = model_folder_path / "model_final.pth"
elif model_type == "ema":
    model_pth_path = model_folder_path / "model_ema_final.pth"
elif model_type == "checkpoint":
    # プロットしたいモデルのエポック数を指定
    epoch_to_plot = 2
    checkpoint_folder_path = pathlib.Path("./models") / trained_model_folder_name / "checkpoints"
    model_pth_path = checkpoint_folder_path / f"checkpoint_epoch_{epoch_to_plot:03d}.pth"


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# モデルのロード
model_config, diffuser_config, train_config = load_config(trained_model_folder_name)
print("Model config:", model_config)
print("Diffuser config:", diffuser_config)
print("Train config:", train_config)
model = build_model(model_config, device)
model.load_state_dict(torch.load(model_pth_path))
diffuser_config["gamma"] = diffusion_guidance_scale
diffuser = build_diffuser(model, diffuser_config, device)

num_labels = train_config["dataset_config"]["num_labels"]

# プロットの準備
plotter = Plotter(dataset_name=train_config["dataset_name"])

In [ ]:
# generate samples
labels = torch.arange(0, num_labels).repeat(2)
images = diffuser.ddim_sampling(x_shape=(20, 3, 32, 32), labels=labels, ddim_timestep=50, eta=0)
plotter.show_images_cond(images, labels)